In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from google.colab import files
import zipfile, os

# Upload du ZIP du code
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

# Extraction dans /content/
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

# Aller là où les fichiers sont réellement
os.chdir('/content/')

# Vérifier ce qui existe
print("Dossier actuel :", os.getcwd())
print("Contenu :", os.listdir('.'))

Saving projet_cnn.zip to projet_cnn.zip
Dossier actuel : /content
Contenu : ['.config', 'drive', 'dataset.py', 'models', 'main.py', 'projet_cnn.zip', 'train.py', 'utils.py', 'sample_data']


In [3]:
from google.colab import files
import zipfile

# Upload du dataset ZIP (~60 Mo, prend ~1 min)
uploaded2 = files.upload()   # sélectionner DIT-FLOWER_DATASET.zip
zip_ds = list(uploaded2.keys())[0]

with zipfile.ZipFile(zip_ds, 'r') as z:
    z.extractall('/content/projet_cnn/')

# Vérifier la structure extraite
import os
for root, dirs, files_list in os.walk('ANSD-FLOWER_DATASET'):
    level = root.replace('ANSD-FLOWER_DATASET', '').count(os.sep)
    print('  ' * level + os.path.basename(root) + '/')
    if level == 1:
        print('  ' * (level+1) + f'{len(files_list)} images')

Saving DIT-FLOWER_DATASET.zip to DIT-FLOWER_DATASET.zip


In [5]:
import shutil, os

# Créer les dossiers data/train/ et data/val/
os.makedirs('data/train/daisy',     exist_ok=True)
os.makedirs('data/train/dandelion', exist_ok=True)
os.makedirs('data/val/daisy',       exist_ok=True)
os.makedirs('data/val/dandelion',   exist_ok=True)

In [7]:
import shutil, os, glob

# D'abord, trouver où le dataset s'est extrait
print("Cherche le dossier daisy...")
results = glob.glob('/content/**/daisy', recursive=True)
print("Trouvé :", results)

Cherche le dossier daisy...
Trouvé : ['/content/projet_cnn/__MACOSX/ANSD-FLOWER_DATASET/train/daisy', '/content/projet_cnn/__MACOSX/ANSD-FLOWER_DATASET/valid/daisy', '/content/projet_cnn/__MACOSX/ANSD-FLOWER_DATASET/test/daisy', '/content/projet_cnn/ANSD-FLOWER_DATASET/train/daisy', '/content/projet_cnn/ANSD-FLOWER_DATASET/valid/daisy', '/content/projet_cnn/ANSD-FLOWER_DATASET/test/daisy', '/content/data/train/daisy', '/content/data/val/daisy']


In [8]:
import shutil, os

BASE = '/content/projet_cnn/ANSD-FLOWER_DATASET'

os.makedirs('data/train/daisy',     exist_ok=True)
os.makedirs('data/train/dandelion', exist_ok=True)
os.makedirs('data/val/daisy',       exist_ok=True)
os.makedirs('data/val/dandelion',   exist_ok=True)

shutil.copytree(f'{BASE}/train/daisy',      'data/train/daisy',     dirs_exist_ok=True)
shutil.copytree(f'{BASE}/train/dandelion',  'data/train/dandelion', dirs_exist_ok=True)
shutil.copytree(f'{BASE}/valid/daisy',      'data/val/daisy',       dirs_exist_ok=True)
shutil.copytree(f'{BASE}/valid/dandelion',  'data/val/dandelion',   dirs_exist_ok=True)

for split in ['train', 'val']:
    for cls in ['daisy', 'dandelion']:
        n = len(os.listdir(f'data/{split}/{cls}'))
        print(f'  data/{split}/{cls} : {n} images')

  data/train/daisy : 529 images
  data/train/dandelion : 746 images
  data/val/daisy : 163 images
  data/val/dandelion : 201 images


In [9]:
!pip install wandb scikit-learn tqdm -q

In [12]:
import wandb
wandb.login("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")   # coller ta clé API depuis wandb.ai/settings



/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: abdoukillmonger (abdoukillmonger-me) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [13]:
# Vérifier le GPU
import torch
print("GPU :", torch.cuda.get_device_name(0))
print("CUDA disponible :", torch.cuda.is_available())

GPU : Tesla T4
CUDA disponible : True


In [15]:
!sed -i 's/, verbose=False//' main.py
# Vérifier que c'est bien retiré :
!grep -n "verbose" main.py

In [16]:
# Lancer l'entraînement complet
!python main.py

  Seeds fixées à 42

  Device : cuda
  GPU    : Tesla T4

  INSPECTION DU DATASET
  Dataset "data/train" : 1275 images | classes : {'daisy': 0, 'dandelion': 1}

──────────────────────────────────────────────────
  Exemples du jeu d'entraînement
──────────────────────────────────────────────────
  Total : 1275 images
    daisy : 529 images (41.5%)
    dandelion : 746 images (58.5%)
  ✓ Classes équilibrées
Figure(1200x600)
  → Sauvegardé : exemples_Exemples_du_jeu_d'entraînement.png

  Classes détectées : ['daisy', 'dandelion']

  PARTIE 1 — LeNet-5 (tanh vs relu vs sigmoid)

  → LeNet5-TANH
  Dataset "data/train" : 1275 images | classes : {'daisy': 0, 'dandelion': 1}
  Dataset "data/val" : 364 images | classes : {'daisy': 0, 'dandelion': 1}
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: abdoukillmonger (abdoukillmonger-me) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boo

In [17]:
from google.colab import files
import glob

# Télécharger tous les PNG et le meilleur checkpoint
for f in glob.glob('*.png'):
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>